In [13]:
import torch
from torch.utils.data import Dataset, DataLoader

class NeedleDataset(Dataset):
    def __init__(self, seq_len, vocab_size=50, n_samples=1000):
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        self.n_samples = n_samples

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        # Random sequence
        seq = torch.randint(2, self.vocab_size, (self.seq_len,))
        print(seq)
        
        # Plant the needle at a random position
        needle_pos = torch.randint(0, self.seq_len, (1,)).item()
        needle_val = 1  # token ID 1 is the "needle"
        seq[needle_pos] = needle_val
        
        # Label: position of the needle
        label = needle_pos
        return seq, label

In [14]:
ds = NeedleDataset(32)


In [17]:
ds[1]
ds[2]


tensor([27, 26, 37, 33, 16,  8, 46, 19, 28, 46, 47, 16, 32, 25, 11, 21, 30, 24,
        47, 38, 41, 29, 36, 23, 29, 39, 31, 35,  7, 19, 37, 29])
tensor([35, 21, 31, 42, 15,  2, 30, 15, 17,  3, 45, 35, 13, 19, 38, 10, 15, 23,
        29, 13, 12,  4, 16, 49, 29, 27, 45, 18, 34, 12, 32, 29])


(tensor([35, 21, 31, 42, 15,  2, 30, 15, 17,  3, 45, 35, 13, 19, 38,  1, 15, 23,
         29, 13, 12,  4, 16, 49, 29, 27, 45, 18, 34, 12, 32, 29]),
 15)

In [16]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, vocab_size=50, embed_dim=64, hidden_dim=128, seq_len=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, seq_len)  # predict needle position

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # take last hidden state
        return self.fc(out)